In [1]:
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
sns_colors = sns.color_palette("tab10")

from pymatgen.core.structure import Structure

from ase.io import read, Trajectory
from ase.visualize import view
from pymatgen.io.ase import AseAtomsAdaptor

from gnome_dreams_oer_screening import OER_catalyst_study

# Rutile TiO2 used for testing:
# TiO2 = read("ZrO2_m.cif")
TiO2 = read("TiO2_rutile.cif")
TiO2_structure = AseAtomsAdaptor.get_structure(TiO2)

# Example GNoME structures contaning Ir:
test_cands = Trajectory('test_cands.traj')
if False:
    view(TiO2)

In [2]:
cand = TiO2 # test_cands[-1]

In [3]:
study = OER_catalyst_study(cand,     
    H2O_gas_free_energy = -14.217,
    H2_gas_free_energy = -6.77
)
study.set_relaxed_structure(cand)

In [4]:
study.identify_distinct_surfaces(max_miller = 1)

In [5]:
study.predict_most_likely_surfaces(max_miller = 1)
df = study.get_termination_rankings()
# df.sort_values('Normalized score', ascending=False)
df

,Miller indices,Termination index,Score,Normalized score
0,"(1, 1, 1)",0,-0.279671,0.021723
1,"(1, 1, 1)",1,-0.279671,0.021723
2,"(1, 1, 0)",2,-0.103895,1.000000
3,"(1, 0, 1)",3,-0.238485,0.250942
4,"(1, 0, 0)",4,-0.146930,0.760490
5,"(0, 0, 1)",5,-0.283574,0.000000


In [6]:
termination_index = 0
if False:
    view(study.get_init_adsorption_sites_atoms(termination_index = 
                                               termination_index))

study.get_init_adsorption_sites_df(termination_index = 
                                   termination_index)

,site type,ad site element,ad site neighboring elements,reduced coordination,position,atom_index,G_approx(O),G(O),G_approx(OH),G(OH),G_approx(OOH),G(OOH),ideal overpotential
1,on-top,Ti,"[(O, 1.9518096083479366), (O, 1.95180960834793...",N/A,"[3.082857722966928, 1.9826999215405692, 15.983...",10.0,None,None,None,None,None,None,None
2,lattice O,N/A,"[(Ti, 1.9518096083479375), (Ti, 3.492789173289...",2,"[0.6983665237530797, 0.14475320546559386, 13.8...",31.0,None,None,None,None,None,None,None


In [7]:
study.initialize_oer_surface_study(termination_index = termination_index)
surface_study = study.get_surface_studies()[termination_index]

In [8]:
atoms = surface_study.get_non_relaxed_surface()
surface_study.set_relaxed_base_surface(atoms, energy = 10)
surface_study.determine_adsorption_sites()
surface_study.get_adsorption_sites_df()

,Site index,site type,ad site element,ad site neighboring elements,reduced coordination,position,atom_index,G_approx(O),G(O),G_approx(OH),G(OH),G_approx(OOH),G(OOH),ideal overpotential
1,1,on-top,Ti,"[(O, 1.951809608347933), (O, 1.951809608347937...",N/A,"[3.082857722966928, 1.9826999215405692, 27.461...",10.0,None,None,None,None,None,None,None
2,2,lattice O,N/A,"[(Ti, 1.951809608347941), (Ti, 3.4927891732897...",2,"[0.6983665237530797, 0.14475320546559386, 25.3...",31.0,None,None,None,None,None,None,None


In [9]:
ad_site_index = 1
surface_study.initialize_adsorption_site_study(ad_site_index)
ad_site_study = surface_study.get_adsorption_site_studies_dict()[ad_site_index]

In [10]:
ad_site_atoms = ad_site_study.get_atoms_for_O_calc()
ad_site_study.set_relaxed_O_calc(ad_site_atoms, energy = 10)

In [11]:
OH_init_atoms_list = ad_site_study.get_OH_adsorbed_atoms()
ad_site_study.set_relaxed_OH_calc(OH_init_atoms_list[0], energy = 10)

In [12]:
surface_study.update_adsorption_energies()
surface_study.get_adsorption_sites_df()

,Site index,site type,ad site element,ad site neighboring elements,reduced coordination,position,atom_index,G_approx(O),G(O),G_approx(OH),G(OH),G_approx(OOH),G(OOH),ideal overpotential
1,1,on-top,Ti,"[(O, 1.951809608347933), (O, 1.951809608347937...",N/A,"[3.082857722966928, 1.9826999215405692, 27.461...",10.0,7.517,None,11.193,None,None,None,9.96
2,2,lattice O,N/A,"[(Ti, 1.951809608347941), (Ti, 3.4927891732897...",2,"[0.6983665237530797, 0.14475320546559386, 25.3...",31.0,None,None,None,None,None,None,None
